# Resnet50 Post Training Quantization

In this notebook, we will use [Quark](https://quark.docs.amd.com/) to perform Post Training Quantization (PTQ) on the Resnet50 model trained for CIFAR10.

## Recommended Hardware

This notebook can run on the following hardware or remote resources.

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/quant/02.a.quantize_resnet_onnx.ipynb)  

## Software Environment

Install ROCm on your system.

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Quantize a CNN using PTQ using Quark
- Validate accuracy with and without calibration

## Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [ ]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/quant/aup_config.py

In [ ]:
from aup_config import aup_setup
aup_setup()

## Quantizing with Quark

### Import Libraries

In [ ]:
import numpy as np
import os

import torch
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision
import torchvision.transforms as transforms

import onnx
import onnxruntime
from onnxruntime.quantization.calibrate import CalibrationDataReader
from onnxruntime.quantization import CalibrationDataReader
from quark.onnx.quantization.config import QConfig
from quark.onnx import ModelQuantizer


[QUARK-INFO]: Checking custom ops library ...

[QUARK-INFO]: The CPU version of custom ops library already exists.

[QUARK-INFO]: The GPU version of custom ops library already exists.

[QUARK-INFO]: Checked custom ops library.


## Prepare the Data

### Download the CIFAR-10 dataset

Execute the following cells to download the CIFAR-10 dataset. We will also create a validation and calibration dataloader. The validation dataloader is used to compute the model accuracy after quantization. The calibration dataloader is used in PTQ.

In [ ]:
data_dir= os.path.join('data')

batch_size = 16

transform = transforms.Compose(
    [transforms.Pad(4), transforms.RandomHorizontalFlip(), transforms.RandomCrop(32), transforms.ToTensor()]
)

val_dataset = torchvision.datasets.CIFAR10(root=data_dir, train=False, transform=transforms.ToTensor(), download=True)

calib_size = 500
val_size = len(val_dataset) - calib_size

# Reproducible split
generator = torch.Generator().manual_seed(0)
calib_subset, val_subset = random_split(val_dataset, [calib_size, val_size], generator=generator)

cal_loader = DataLoader(dataset=calib_subset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(dataset=val_subset, batch_size=1, shuffle=False)

The CIFAR10 classes are enumerated as follow

In [4]:
classes = [v for _,v in enumerate(val_dataset.classes)]
for idx, v in enumerate(classes):
    print(f'{idx}: {v}')

0: airplane
1: automobile
2: bird
3: cat
4: deer
5: dog
6: frog
7: horse
8: ship
9: truck


## Quantize the Model

We are going to use a finetuned ResNet-50 model for the CIFAR-10 dataset. If you would like to learn how to finetune a pre-trained model for the CIFAR10 dataset, check this notebook [here](./02.b.fine_tune_resnet50.ipynb).


The aim of this sections is to quantize the model using PTQ with and without a calibration dataset.

### PTQ Uncalibrated quantization

Quantizing AI models from floating-point to 8-bit integers reduces computational power and the memory footprint required for inference. 
For model quantization, there are a number of frameworks that perform this task. This example uses the [AMD Quark](https://quark.docs.amd.com/latest/index.html) quantizer workflow.

Let us start by defining the original model source and the destination of the quantized models.

In [ ]:
input_model_path = "onnx/resnet_trained_for_cifar10.onnx"
output_model_uncal_path = "onnx/resnet.uncal.qdq.U8S8.onnx"
output_model_cal_path = "onnx/resnet.cal.qdq.U8S8.onnx"

We will use the `QConfig.get_default_config` to get one of the default quantization configurations. [List of default quantization configurations](https://quark.docs.amd.com/latest/onnx/user_guide_config_description.html#more-quantization-default-configurations).

In this case, we are using `INT8_CNN_DEFAULT`. We will also set `UseRandomData` to true not to use calibration. Finally, we setup the log severity to the less verbose mode.

In [ ]:
quant_name_config = "INT8_CNN_DEFAULT"
quantization_config_uncal = QConfig.get_default_config(quant_name_config)
quantization_config_uncal.global_quant_config.log_severity_level = 5
quantization_config_uncal.global_quant_config.extra_options['UseRandomData'] = True

With the quantization configuration setup, we can now create the `ModelQuantizer` object. Using this object, we call the `quantize_model` function passing the original ONNX model and the destination model.

This will generate a quantized model using QDQ quant format and UInt8 activation type and Int8 weight type. After the run is completed, the quantized ONNX model is saved to `onnx/resnet.uncal.qdq.U8S8.onnx`.
    
For more information on representation of quantized ONNX models (e.g., QDQ quant format, UInt8 activation type and Int8 weight type) see [here](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html#onnx-quantization-representation-format)


In [7]:
quantizer_uncal = ModelQuantizer(quantization_config_uncal)
quantizer_uncal.quantize_model(input_model_path, output_model_uncal_path)

print('Uncalibrated and quantized model saved at:', output_model_uncal_path)


[QUARK-WARNING]: Config has been replaced by QConfig. The old API will be removed in the next release.


[QUARK_INFO]: Time information:
2025-11-04 18:00:07.157160
[QUARK_INFO]: OS and CPU information:
                                        system --- Linux
                                          node --- b9b02bfa05a0
                                       release --- 6.14.0-33-generic
                                       version --- #33~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Sep 19 17:02:30 UTC 2
                                       machine --- x86_64
                                     processor --- x86_64
[QUARK_INFO]: Tools version information:
                                        python --- 3.12.3
                                          onnx --- 1.19.0
                                   onnxruntime --- 1.22.1
                                    quark.onnx --- 0.10+b8ad5c1d29
[QUARK_INFO]: Quantized Configuration information:
                                   model_input --- onnx/resnet_trained_for_cifar10.onnx
                                  model_output --- onnx/resn

100%|██████████| 125/125 [00:00<00:00, 5438.28it/s]


┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Op Type              ┃ Float Model                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Conv                 │ 53                              │
│ Relu                 │ 50                              │
│ MaxPool              │ 1                               │
│ Add                  │ 16                              │
│ GlobalAveragePool    │ 1                               │
│ Flatten              │ 1                               │
│ Gemm                 │ 2                               │
├──────────────────────┼─────────────────────────────────┤
│ Quantized model path │ onnx/resnet.uncal.qdq.U8S8.onnx │
└──────────────────────┴─────────────────────────────────┘

┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ Op Type           ┃ Activation ┃ Weights  ┃ Bias     ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ Conv              │ UINT8(53)  │ INT8(53) │ INT8(53) │
│ MaxPool           │ UINT8(1)   │          │          │
│ Add               │ UINT8(16)  │          │          │
│ GlobalAveragePool │ UINT8(1)   │          │          │
│ Flatten           │ UINT8(1)   │          │          │
│ Gemm              │ UINT8(2)   │ INT8(2)  │ INT8(2)  │
└───────────────────┴────────────┴──────────┴──────────┘

Uncalibrated and quantized model saved at: onnx/resnet.uncal.qdq.U8S8.onnx


### PTQ Calibrated Quantization

PTQ typically losses accuracy after the model is quantized, however, we can gain back some of the lost accuracy by providing a calibration dataset. These are representative inputs provided during the quantization to help with finding the correct quantization values.

To pass a calibration dataset to Quark, we need an object with the `get_next` method and a way of accessing to an individual item. The following cells creates the `cal_dr`, using the calibration subset we defined at the beginning of the notebook.

In [ ]:
class PytorchResNetDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        input_data = sample[0]
        label = sample[1]
        return input_data, label

    
class ResnetCalibrationDataReader(CalibrationDataReader):
    def __init__(self, batch_size: int = 16):
        super().__init__()
        self.iterator = iter(DataLoader(PytorchResNetDataset(calib_subset), batch_size=batch_size))

    def get_next(self) -> dict:
        try:
            images, labels = next(self.iterator)
            return {"input": images.numpy()}
        except Exception:
            return None


cal_dr = ResnetCalibrationDataReader()

Now that we have the calibration set, we can create the quantization configuration. As before, we will use the `XINT8`, we unset the `UseRandomData` to force Quark to use the calibration set. Finally, we setup the log severity to the less verbose mode.

In [ ]:
quantization_config_cal = QConfig.get_default_config(quant_name_config)
quantization_config_cal.global_quant_config.extra_options['UseRandomData'] = False
quantization_config_cal.global_quant_config.log_severity_level = 5

Now create the `ModelQuantizer` object. Using this object, we call the `quantize_model` function passing the original ONNX model and the destination model.

This will generate a quantized model using QDQ quant format and UInt8 activation type and Int8 weight type. After the run is completed, the quantized ONNX model is saved to `onnx/resnet.cal.qdq.U8S8.onnx`.

In [ ]:
quantizer_cal = ModelQuantizer(quantization_config_uncal)
quantizer_cal.quantize_model(input_model_path, output_model_cal_path, cal_dr)

print('Calibrated and quantized model saved at:', output_model_cal_path)

[QUARK_INFO]: Time information:
2025-11-04 18:00:11.045647
[QUARK_INFO]: OS and CPU information:
                                        system --- Linux
                                          node --- b9b02bfa05a0
                                       release --- 6.14.0-33-generic
                                       version --- #33~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Sep 19 17:02:30 UTC 2
                                       machine --- x86_64
                                     processor --- x86_64
[QUARK_INFO]: Tools version information:
                                        python --- 3.12.3
                                          onnx --- 1.19.0
                                   onnxruntime --- 1.22.1
                                    quark.onnx --- 0.10+b8ad5c1d29
[QUARK_INFO]: Quantized Configuration information:
                                   model_input --- onnx/resnet_trained_for_cifar10.onnx
                                  model_output --- onnx/resn

100%|██████████| 125/125 [00:02<00:00, 45.38it/s]


┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Op Type              ┃ Float Model                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Conv                 │ 53                            │
│ Relu                 │ 50                            │
│ MaxPool              │ 1                             │
│ Add                  │ 16                            │
│ GlobalAveragePool    │ 1                             │
│ Flatten              │ 1                             │
│ Gemm                 │ 2                             │
├──────────────────────┼───────────────────────────────┤
│ Quantized model path │ onnx/resnet.cal.qdq.U8S8.onnx │
└──────────────────────┴───────────────────────────────┘

┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ Op Type           ┃ Activation ┃ Weights  ┃ Bias     ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ Conv              │ UINT8(53)  │ INT8(53) │ INT8(53) │
│ MaxPool           │ UINT8(1)   │          │          │
│ Add               │ UINT8(16)  │          │          │
│ GlobalAveragePool │ UINT8(1)   │          │          │
│ Flatten           │ UINT8(1)   │          │          │
│ Gemm              │ UINT8(2)   │ INT8(2)  │ INT8(2)  │
└───────────────────┴────────────┴──────────┴──────────┘

Calibrated and quantized model saved at: onnx/resnet.cal.qdq.U8S8.onnx


## Evaluate the Quantized Models

Let us see how the quantized models perform in term of accuracy. The next function, takes the model as an argument. Then loads the model using `onnxruntime`, note that the model is run on the CPU. Then, using the validation dataloader, we iterate image by image and run the inference. We count the number of correct classification and the total number of classifications. With this, we can compute the accuracy.

In [10]:
def compute_accuracy(model) -> float:
    session = onnxruntime.InferenceSession(model.SerializeToString(), providers=['CPUExecutionProvider'])

    correct = 0
    total = 0

    for idx, (images, labels) in enumerate(val_loader):
        input_data = images.cpu().detach().numpy()
        outputs = session.run(None, {'input': input_data})

        predicted_class = np.argmax(outputs[0])
        total += 1
        correct += (torch.tensor(predicted_class) == labels.squeeze()).sum().item()

    del session
    return 100 * correct / total

Compute accuracy for uncalibrated quantized model

In [11]:
accuracy_uncal = compute_accuracy(onnx.load(output_model_uncal_path))

Compute accuracy for calibrated quantized model

In [12]:
accuracy_cal = compute_accuracy(onnx.load(output_model_cal_path))

Comparing the results you can see that the calibrated model has gained back 1% of the accuracy.

In [13]:
print(f" Accuracy of the uncalibrated quantized model : {accuracy_uncal:.2f} % validated on {len(val_subset)} images")
print(f" Accuracy of the calibrated quantized model : {accuracy_cal:.2f} % validated on {len(val_subset)} images")

 Accuracy of the uncalibrated quantized model : 83.42 % validated on 9500 images
 Accuracy of the calibrated quantized model : 83.52 % validated on 9500 images


## Models Size

Now, let us look at the size of the original and quantized models. We are going to read the size directly from the saved files in disk.

In [ ]:
model_sizes = {}
for m in [input_model_path, output_model_uncal_path, output_model_cal_path]:
    model_sizes[m] = {'size': os.path.getsize(m)}
    print(f"model: {m} size: {model_sizes[m]['size']/1024/1024:.2f} MiB") 

Now that we have the size for the 3 models, you can clearly see a significant reduction in size for the quantized models. Let us find out that ratio. 

You can see that we will get almost 4x reduction, which is expected as we go from 32-bit number representation to a 8-bit representation. There is a bit of overhead in the format itself, that is why we do not see the 4x.

In [ ]:
size_ratio = model_sizes[input_model_path]['size'] / model_sizes[output_model_cal_path]['size']
print(f'{size_ratio=:.2f}')

## Exercises for the Reader

In the notebook, we showed you PTQ with and without calibration for an `INT8_CNN_DEFAULT` configuration. We invite you to complete the table below by using different quantization configurations.

| Quant Type        | Uncalibrate | Calibrated |
|:-----------------:|:-----------:|:----------:|
| XINT8             |      |     |
| INT8_CNN_DEFAULT  | 85.68 %     | 86.62 %    |
| S8S8_AAWS         |      |     |
| U8S8_AAWS         |      |     |
| S16S8_ASWS        |      |     |
| A8W8              |      |     |
| A16W8             |      |     |

## References

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://quark.docs.amd.com/">Quark</a></li>
</ul>
</div>

## Conclusions

In this notebook, you use Quark to perform Post Training Quantization on a pre-trained model. You explored different quantization schemes and analyze accuracy implications. You also compared the model size after quantization.

---

[AMD University Program](https://www.amd.com/aup).

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT